# Example Literature Project

## Helper Functions

In [ ]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(
        Path(project_dir),
        conversation_path=path,
        router_model="Qwen/Qwen3-4B-MLX-4bit",
    )
    print(f"Conversation: {path}")
    return store, conversation, session





### GUI-equivalent decision callbacks

In [ ]:
def _show_gui_choice(question, options, selected):
    """Print the offered options and the scripted GUI selection.

    Parameters
    ----------
    question : str
        Tool question shown by the desktop GUI.
    options : tuple of str
        Choices offered by the tool.
    selected : int or None
        Chosen zero-based index, or None for cancellation.

    """
    print(f"GUI choice requested: {question}")
    for index, option in enumerate(options):
        marker = "*" if index == selected else " "
        print(f"  {marker} {index}: {option}")
    if selected is None:
        print("  Decision: cancel (no scripted choice matched)")
    else:
        print(f"  Decision: option {selected}")


def choose_first_20(question, options):
    """Choose the GUI's first-20-items chat display, when offered.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Matching option index, or None when not offered.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_save_all(question, options):
    """Choose the GUI option to save all matches as Markdown.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Export option index, or None when not offered.

    """
    selected = next(
        (
            i for i, value in enumerate(options)
            if value.startswith("Save all ")
            and value.endswith(" to outputs/ as Markdown")
        ),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_gui_default(question, options):
    """Use a documented automatic selection for common GUI choices.

    Long lists use the GUI's 20-item display. An ambiguous document
    question chooses the first offered paper, as if the user clicked the
    first GUI option. Every decision is printed; other choices cancel.

    Parameters
    ----------
    question : str
        Tool question.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Selected option index, or None to cancel.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    if selected is None and options and (
        "which paper" in question.casefold()
        or "which document" in question.casefold()
    ):
        selected = 0
    _show_gui_choice(question, options, selected)
    return selected


def choose_document_named(filename):
    """Create a callback that selects one named file in a GUI prompt.

    Parameters
    ----------
    filename : str
        Filename to select, including its extension.

    Returns
    -------
    callable
        Callback with the GUI's ``(question, options)`` signature.

    """
    def choose(question, options):
        """Select the requested file from the currently offered options."""
        selected = next(
            (
                index for index, value in enumerate(options)
                if value.split(" (", 1)[0].casefold()
                == filename.casefold()
            ),
            None,
        )
        _show_gui_choice(question, options, selected)
        return selected

    return choose


def _confirm_gui_plan(plan, selection):
    """Report a GUI-equivalent plan decision, disallowing web access.

    Parameters
    ----------
    plan : ToolPlan
        Plan offered by the tool.
    selection : str
        Intended choice: continue, quick, or cancel.

    Returns
    -------
    str
        Applied choice, with network requests always cancelled.

    """
    chosen = (
        "cancel" if getattr(plan, "operation", "model") == "network"
        else selection
    )
    print(f"GUI plan: {plan.tool} | {plan.summary}")
    print(
        f"  Items: {plan.items}, batches: {plan.batches_to_run}, "
        f"quick items: {plan.quick_items}"
    )
    print(f"  Decision: {chosen}")
    return chosen


def confirm_gui_continue(plan):
    """Simulate clicking Continue, but never approve network lookup."""
    return _confirm_gui_plan(plan, "continue")


def confirm_gui_quick(plan):
    """Simulate clicking Quick, but never approve network lookup."""
    return _confirm_gui_plan(plan, "quick")


def confirm_gui_cancel(plan):
    """Simulate clicking Cancel on a tool's proposed plan."""
    return _confirm_gui_plan(plan, "cancel")



### Question and trace inspection

In [ ]:
def ask_question(
    session,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_continue,
):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.
    choice_callback : callable or None, optional
        Select a GUI tool option. The default displays the first 20
        matches or selects the first offered paper. None deliberately
        uses the noninteractive fallback.
    confirm_callback : callable or None, optional
        Select Continue, Quick, or Cancel for an expensive tool plan.
        The default selects Continue; network plans always cancel.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(
            question,
            retrieval_count=10,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    if session.turns:
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        print(f"Resolved intent: {parameters.get('intent', 'unknown')}")
        if isinstance(tool, dict) and tool.get("name"):
            print(f"Executed tool: {tool['name']}")
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def _print_retrieved_chunks(stages, width=79):
    """Display retrieved passages recorded in the native trace.

    Parameters
    ----------
    stages : dict
        Stages of one recorded conversation turn.
    width : int, optional
        Maximum display width.

    """
    retrieval = stages.get("03_retrieval", {})
    operations = retrieval.get("operations", {})

    print("\nRETRIEVED CHUNKS")

    found = 0

    def visit(value, path):
        nonlocal found

        if isinstance(value, dict):
            text = next(
                (
                    value[key]
                    for key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    )
                    if isinstance(value.get(key), str) and value[key].strip()
                ),
                None,
            )

            if text is not None:
                found += 1
                print(f"\n  CHUNK {found}")
                print(f"  Trace location: {path}")

                for key, item in value.items():
                    if key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    ):
                        continue

                    if isinstance(item, (str, int, float, bool)):
                        _print_wrapped(f"  {key}", item, width)

                _print_wrapped("  PASSAGE", text, width)
                return

            for key, child in value.items():
                visit(child, f"{path}/{key}")

        elif isinstance(value, list):
            for index, child in enumerate(value):
                visit(child, f"{path}/{index}")

    for name, operation in operations.items():
        if "retrieval" not in name.lower():
            continue

        visit(operation.get("details", {}), name)

    if not found:
        print("  No passage text found in the retrieval trace.")


def inspect_conversation(
    conversation_path,
    turn_number=None,
    show_chunks=True,
):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.
    show_chunks : bool, optional
        Whether to display passages recorded in the retrieval trace.
        Defaults to True.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        """Find a stage operation even if its sequence number changed."""
        operations = stages.get(stage_name, {}).get("operations", {})
        exact = operations.get(operation_name)
        if exact is not None:
            return exact
        suffix = "_" + operation_name.split("_", 1)[1]
        return next(
            (entry for key, entry in operations.items()
             if key.endswith(suffix)),
            {},
        )

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields(
            "INTENT",
            decision.get("classification", {}),
            ["intent", "standalone", "refers_back", "related_turn_ids"],
        )
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields(
            "RESOLVED POLICY",
            decision.get("resolved", {}),
            ["history_policy", "retrieval_mode", "retrieval_strategy"],
        )
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields(
            "EFFECTIVE SETTINGS",
            settings,
            [
                "project_type_id",
                "backend",
                "chat_model",
                "router_model",
                "thinking",
                "temperature",
                "max_new_tokens",
                "retrieval_count",
            ],
        )
        retrieval = operation(stages, "03_retrieval", "02_retrieval_result")
        show_fields(
            "RETRIEVAL",
            retrieval.get("details", {}),
            ["mode", "strategy", "evidence_count", "duration_ms"],
        )
        if show_chunks:
            _print_retrieved_chunks(stages)

        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields(
                        "  DETAILS",
                        entry.get("details", {}),
                        ["tool", "tool_name", "status", "duration_ms"],
                    )
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(
                f"MODEL CALL: {stage_name}/{name}",
                details,
                [
                    "call_id",
                    "model_id",
                    "backend",
                    "device",
                    "thinking",
                    "input_token_count",
                    "prompt_token_count",
                    "output_token_count",
                    "finish_reason",
                    "reply_duration_ms",
                ],
            )
            show_fields(
                "GENERATION SETTINGS",
                details.get("generation", {}),
                ["temperature", "max_new_tokens", "thinking"],
            )

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace




### Intent and tool verification helpers

In [ ]:
# Intent coverage is based on saved turn metadata, not on the wording of
# the test question or the route we expected the model to choose.
intent_checks = []


def _intent_definitions():
    """Read the active repository's literature intent definitions."""
    path = repo_root / "kilin" / "project_types" / "literature.json"
    with path.open(encoding="utf-8") as file:
        return json.load(file)["intents"]


# Fields demonstrating that a tool progressed beyond initial selection.
# They are deliberately not assertions about scientific correctness.
_TOOL_RESULT_FIELDS = {
    "project_inventory": ("files",),
    "scan_and_list": ("total_mentions", "listed"),
    "show_source": ("turn", "passages"),
    "filter_results": ("turn", "items"),
    "paper_overview": ("paper",),
    "suggest_reference": ("statement", "papers_checked"),
    "find_passages": ("topic", "choice"),
    "find_fact": ("choice",),
    "batch_review": ("document", "choice"),
    "batch_classify": ("categories", "choice"),
    "reference_check": ("paper",),
    "generate_paper_summary": ("template", "saved"),
}


def check_intent(
    session,
    expected_intent,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_quick,
):
    """Execute a test and compare saved routing to its intended path.

    Parameters
    ----------
    session : RAGSession
        Active session; use a new conversation for independent groups.
    expected_intent : str
        Intended name in ``literature.json``; never forced on the router.
    question : str
        Test request sent unchanged to KíLín.
    choice_callback : callable or None, optional
        Scripted GUI choice. The default shows lists in the chat.
    confirm_callback : callable or None, optional
        Scripted GUI confirmation. Quick is the default to limit work.

    Returns
    -------
    str or None
        Answer, or None when this test failed before an answer was saved.

    """
    definitions = _intent_definitions()
    if expected_intent not in definitions:
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    expected_tool = definitions[expected_intent].get("tool", {}).get("name")
    count_before = len(session.turns)
    record = {
        "expected": expected_intent,
        "expected_tool": expected_tool,
        "question": question,
    }
    try:
        answer = ask_question(
            session,
            question,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
        if len(session.turns) != count_before + 1:
            raise RuntimeError("The question did not save exactly one turn.")
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        actual_tool = tool.get("name") if isinstance(tool, dict) else None
        record.update(
            actual=parameters.get("intent"),
            actual_tool=actual_tool,
            invalid_citations=parameters.get("invalid_citations", []),
        )
        record["status"] = (
            "MATCH"
            if record["actual"] == expected_intent
            and actual_tool == expected_tool
            else "MISMATCH"
        )
        if isinstance(tool, dict):
            record["output"] = tool.get("saved") or tool.get("report")
            required = _TOOL_RESULT_FIELDS.get(expected_tool, ())
            missing = [key for key in required if key not in tool]
            if record["status"] == "MATCH" and missing:
                record["status"] = "INCOMPLETE"
                record["missing"] = missing
            # Save and report cases should create a file if advertised.
            saved = tool.get("saved")
            if saved and not (project_dir / saved).is_file():
                record["status"] = "INCOMPLETE"
                record["missing_file"] = saved
    except Exception as error:
        answer = None
        record["status"] = "ERROR"
        record["error"] = f"{type(error).__name__}: {error}"
        print(f"INTENT TEST ERROR: {record['error']}")
    intent_checks.append(record)
    print(
        f"INTENT TEST: {expected_intent} -> {record.get('actual', '-')}; "
        f"tool {expected_tool or '-'} -> "
        f"{record.get('actual_tool') or '-'}; {record['status']}"
    )
    if record.get("missing"):
        print(f"  Missing tool result fields: {record['missing']}")
    if record.get("missing_file"):
        print(f"  Missing output file: {record['missing_file']}")
    if record.get("invalid_citations"):
        print(f"  Invalid citations: {record['invalid_citations']}")
    if record.get("output"):
        print(f"  Output file: {record['output']}")
    return answer


def skip_intent(expected_intent, reason):
    """Record a test intentionally not executed in the current run."""
    if expected_intent not in _intent_definitions():
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    intent_checks.append(
        {"expected": expected_intent, "status": "SKIPPED",
         "reason": reason}
    )
    print(f"SKIPPED {expected_intent}: {reason}")


def show_intent_coverage():
    """Report attempted coverage and actual tool/intent agreement.

    Returns
    -------
    dict
        Summary counts: matched, mismatched, errors, skipped, untested.

    """
    definitions = _intent_definitions()
    counts = {key: 0 for key in (
        "matched", "mismatched", "incomplete", "errors",
        "skipped", "untested"
    )}
    print("\nLITERATURE INTENT COVERAGE")
    print(f"{'Intent':<27} {'Expected tool':<24} Status")
    print("-" * 79)
    for intent, settings in definitions.items():
        records = [
            entry for entry in intent_checks
            if entry["expected"] == intent
        ]
        active = [entry for entry in records
                  if entry["status"] != "SKIPPED"]
        if not records:
            status = "NOT RUN"
            counts["untested"] += 1
        elif not active:
            status = "SKIPPED"
            counts["skipped"] += 1
        elif all(entry["status"] == "MATCH" for entry in active):
            status = "MATCH"
            counts["matched"] += 1
        elif any(entry["status"] == "ERROR" for entry in active):
            status = "ERROR"
            counts["errors"] += 1
        elif any(entry["status"] == "INCOMPLETE" for entry in active):
            status = "INCOMPLETE"
            counts["incomplete"] += 1
        else:
            status = "MISMATCH"
            counts["mismatched"] += 1
        expected_tool = settings.get("tool", {}).get("name") or "model"
        print(f"{intent:<27} {expected_tool:<24} {status}")
    print("-" * 79)
    print(f"Routing/tool matches: {counts['matched']}/{len(definitions)}")
    print("Note: matching a route does not validate scientific correctness.")
    return counts



## Create Empty Literature Project

In [ ]:
from pathlib import Path

from kilin.workspace import prepare_sources_workspace

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError(
            "Run this notebook from within the KíLín repository."
        )
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
    settings={"allow_download": False},
)

print(f"Created literature project: {project.root}")
print(f"Source directory: {project.source}")

## First Conversation without Sources

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, "What is Brownian motion?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, "Summarize the articles in this project.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    "Find articles about neural networks in this project.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
session.close()

## Add Articles to Project

In [ ]:
from kilin.workspace import add_project_sources

fixtures = repo_root / "tests" / "fixtures"

articles = sorted(fixtures.glob("*.pdf"))

if not articles:
    raise FileNotFoundError(f"No PDF articles found in {fixtures}")

project = add_project_sources(
    project_dir,
    articles,
)

print(f"Project: {project.root}")
print(f"Imported articles: {len(articles)}")

for article in sorted(project.source.glob("*.pdf")):
    print(f"  {article.name}")

## Document Discovery and Retrieval

In [ ]:
store, conversation, session = create_conversation(project_dir)

### GUI-equivalent choices for long mention lists

The next two calls explicitly select the GUI's **Show first 20** and
**Save all** options. A third call disables the callback and exercises
the notebook-only `MAX_LISTED` fallback (100). The selected output limit
does not change the total number of matching mentions.


In [ ]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
# GUI choice: display the first 20 mentions.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_first_20,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


In [ ]:
# GUI choice: save the complete list as markdown.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_save_all,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


In [ ]:
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=None,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


In [ ]:
answer = ask_question(
    session,
    "Show me the source information for the BRAPH 2 article.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    "Find passages discussing graph measures in the BRAPH articles.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
answer = ask_question(
    session,
    "What do those passages say about their implementation?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)

In [ ]:
answer = ask_question(session, "Which articles discuss deep learning?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)

In [ ]:
session.close()

## Article Understanding and Synthesis

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, "Give me an overview of the BRAPH 2 paper.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(
    session,
    "Summarize its main contributions and limitations.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(
    session,
    "Generate a detailed summary of the BRAPH 2 paper.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(
    session,
    "How does BRAPH 2 differ from the earlier BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, "What evidence supports those differences?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    "Which of the four papers is most relevant to graph-based analysis "
    "of brain connectivity, and why?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
session.close()

## Specialized Literature Tools

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(
    session,
    "Find the passage in the BRAPH 2 paper "
    "describing its software architecture.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(
    session,
    "What programming language is BRAPH 2 implemented in?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, "List the articles that mention graph theory.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(
    session,
    "Filter those results to articles discussing brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(
    session,
    "Classify the five articles by their main research topic.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    "Review the five articles for their relevance to a literature review "
    "on computational brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
answer = ask_question(
    session,
    "Check whether the BRAPH 2 paper supports the claim "
    "that it provides a graphical user interface.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)

In [ ]:
answer = ask_question(
    session,
    "Which article should I cite for the introduction of the BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)

In [ ]:
session.close()

## Continuity and Robustness

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, "Summarize the BRAPH 2 article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, "What are its main limitations?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, "How does it compare with the 2017 paper?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(session, "Now tell me about the deep learning article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, "What methods does it use?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(session, "Compare it with the previous article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
answer = ask_question(
    session,
    "Find evidence that all five articles use the same dataset.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)

In [ ]:
answer = ask_question(session, "Which article did we discuss first?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)

In [ ]:
session.close()

## Literature Intent and Tool Coverage

This section verifies **routing and tool selection** for all 21 intents
configured in `kilin/project_types/literature.json`. The test harness
**does not force** an intent: it records the actual intent and tool saved
by `RAGSession.ask()` and prints `MATCH`, `MISMATCH`, or `ERROR`.

The GUI's **choice** and **confirmation** adapters are simulated using
explicit callbacks. The default choice shows up to 20 results in chat;
if asked to choose an ambiguous paper it selects the first offered
paper and prints that decision. The default confirmation for these
coverage tests is **Quick**, to keep batches manageable. Network lookups
are **always declined**, even if Continue is selected for model plans.

Run the groups in order so their contextual follow-ups have evidence.
The extended group (whole-paper scans, classification, reference checks,
and the four output templates) is opt-in because it can take many model
calls and write files. Set `RUN_EXTENDED_CASES = True` to exercise those
nine remaining intents. Inspect each mismatch rather than assuming a
successful answer means the intended operation ran.

A `MATCH` only validates intent and tool selection, **not** the truth
of a generated claim, citation support, report quality, or GUI rendering.
The notebook does not provide the GUI's optional Qt PDF writer; report
tools will still create their Markdown output.


In [ ]:
# False: run the initial 12 intent cases; report the rest as SKIPPED.
# True: exercise every configured literature intent and its selected tool.
RUN_EXTENDED_CASES = False

# Static coverage contract: fail if the project's declared intents change.
PLANNED_INTENTS = {
    "small_talk", "question_answering", "compare_sources",
    "report_generation", "conversation_followup", "rephrase_previous",
    "project_inventory", "mention_listing", "refine_results",
    "source_context", "paper_overview", "suggest_reference",
    "topic_listing", "fact_finding", "document_reading",
    "mention_classification", "reference_check", "research_briefing",
    "popular_science_summary", "technical_summary",
    "experimental_protocol",
}
assert PLANNED_INTENTS == set(_intent_definitions()), (
    "Update the test cases when literature.json adds or removes intents."
)
print(f"Planned intent tests: {len(PLANNED_INTENTS)}")


### A. Direct answers, comparison, and conversation continuity

These six tests share one conversation so the follow-up and rephrasing
intents have actual preceding answers to use. They test routing, not
whether the answer's scientific content is correct.


In [ ]:
store, conversation, session = create_conversation(project_dir)

check_intent(session, "small_talk", "hello")
check_intent(
    session,
    "question_answering",
    "According to the BRAPH 2 paper, why was its software developed?",
)
check_intent(
    session,
    "compare_sources",
    "Compare the methods of mijalkov17braph.pdf and "
    "tiunn26braph2.pdf, including their differences.",
)
check_intent(
    session,
    "report_generation",
    "Write a structured report with headings about methods for "
    "brain connectivity analysis across these papers.",
)
check_intent(
    session,
    "conversation_followup",
    "In your previous report, what was the main limitation you noted?",
)
check_intent(
    session,
    "rephrase_previous",
    "Rewrite your previous answer in two concise bullet points, "
    "without adding any information.",
)

session.close()


### B. File discovery, exact mentions, and prior-result tools

`refine_results` and `source_context` operate on saved evidence from
earlier turns. The 20-item callback matches the GUI's chat-list option.
If retrieval finds no usable mentions, inspect the corresponding tool's
result before interpreting a later failure as a routing defect.


In [ ]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "project_inventory",
    "Which PDF files are available in this project?",
)
check_intent(
    session,
    "mention_listing",
    "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
    choice_callback=choose_first_20,
)
check_intent(
    session,
    "source_context",
    "Show the exact source text and surrounding context for the "
    "first cited sentence in that list.",
)
check_intent(
    session,
    "refine_results",
    "Of the BRAPH mentions you listed, which ones describe "
    "software features rather than bibliographic citations?",
)

session.close()


### C. Paper overview, reference suggestion, and document selection

The third test intentionally omits the filename to exercise the GUI's
paper-selection dialog. The callback chooses `tiunn26braph2.pdf` by
its **actual option label**, not by assuming a fixed index. If the file
is not offered, it cancels instead of picking an unrelated paper.


In [ ]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "paper_overview",
    "Give me a short overview of tiunn26braph2.pdf.",
)
check_intent(
    session,
    "suggest_reference",
    "Suggest a reference from this project for the statement: "
    "BRAPH supports graph analysis of brain networks.",
)
# Additional selection-path test; still expects paper_overview.
check_intent(
    session,
    "paper_overview",
    "Give me a short overview of one paper in the collection.",
    choice_callback=choose_document_named("tiunn26braph2.pdf"),
)

session.close()


### D. Extended tool tests (opt-in)

Enable `RUN_EXTENDED_CASES` in the setup cell to execute the nine
remaining intents. **Quick** is selected for model-intensive plans;
this verifies the quick route, not exhaustive full-processing behavior.
Reference lookup over the network is always denied. These calls may
still be slow, and report templates create Markdown files in `outputs/`.


In [ ]:
extended_cases = (
    (
        "topic_listing",
        "Which papers apply graph-theory methods to analyze brain "
        "connectivity, based on their meaning rather than keywords?",
    ),
    (
        "fact_finding",
        "Give a complete list of the programming languages and software "
        "dependencies named in tiunn26braph2.pdf.",
    ),
    (
        "document_reading",
        "Read the whole tiunn26braph2.pdf paper from start to end "
        "and identify which software components it describes.",
    ),
    (
        "mention_classification",
        "Classify the mentions of BRAPH in tiunn26braph2.pdf into "
        "(1) descriptions of current software and (2) references "
        "to earlier work, then count each group.",
    ),
    (
        "reference_check",
        "Check the references cited by tiunn26braph2.pdf and determine "
        "which cited papers are present in this project.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in extended_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in extended_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


### E. Four structured report tools (opt-in)

Each case targets a distinct intent but calls the same
`generate_paper_summary` tool with a different template. Markdown
reports are saved in the project workspace. Repeating the cells may
create additional output files. The notebook does not generate PDFs.


In [ ]:
report_cases = (
    (
        "research_briefing",
        "Prepare a research briefing for tiunn26braph2.pdf.",
    ),
    (
        "popular_science_summary",
        "Write a popular science summary of tiunn26braph2.pdf "
        "for a general audience.",
    ),
    (
        "technical_summary",
        "Prepare a detailed technical summary of tiunn26braph2.pdf "
        "for specialists.",
    ),
    (
        "experimental_protocol",
        "Extract an experimental protocol based on tiunn26braph2.pdf; "
        "identify missing replication details without inventing them.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in report_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in report_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


### F. Confirmation and output-path regression (optional)

The main test groups choose **Quick** for large model plans. To test
GUI-style cancellation without an expensive scan, run the cell below:
it should produce a cancelled-tool answer without starting model batches.
The ordinary earlier BRAPH cells already exercise **show 20**, **save
all**, and the deliberately noninteractive **show up to 100** paths.


In [ ]:
# Optional: the fact_finding intent always offers a plan when items exist.
# This tests a GUI Cancel choice; it is not included in matched coverage.
RUN_CANCEL_REGRESSION = False

if RUN_CANCEL_REGRESSION:
    store, conversation, session = create_conversation(project_dir)
    try:
        check_intent(
            session,
            "fact_finding",
            "List the programming languages in tiunn26braph2.pdf.",
            confirm_callback=confirm_gui_cancel,
        )
    finally:
        session.close()


### G. Coverage summary

Run this cell after the groups above. A `MATCH` means the **saved**
intent equals the intended intent and the saved tool name equals the
tool defined in `literature.json`. `INCOMPLETE` means routing matched, but the tool did not produce its expected result metadata. `SKIPPED` means the opt-in test was
not executed. If a route mismatches, inspect that question's trace and
revise the example or investigate the router; do not force the intent.


In [ ]:
coverage = show_intent_coverage()

# Full selection-path coverage also requires testing Continue and Cancel
# branches on appropriate non-network tool plans. The automated suite
# above focuses on the ordinary and Quick paths.
